# Baseline Model

Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import random
from collections import defaultdict
import numpy as np


Datasets

In [2]:
games_df = pd.read_pickle('games_processed.pkl')
recommendations_df = pd.read_pickle('recommendations_processed.pkl')

print("Games DataFrame Shape:", games_df.shape)
print("Recommendations DataFrame Shape:", recommendations_df.shape)

Games DataFrame Shape: (4146, 7)
Recommendations DataFrame Shape: (1034570, 8)


Matrix Formation

In [3]:
content_features = games_df['tags'].fillna('')
numeric_features = games_df[['price_final', 'rating', 'user_reviews']]

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(content_features)
combined_features = hstack([tfidf_matrix, numeric_features])
similarity_matrix = cosine_similarity(tfidf_matrix)

Simple Content-Based System

In [4]:
def get_content_recommendations(game_id, similarity_matrix, games_df, n_recommendations=5):
    game_idx = games_df[games_df['app_id'] == game_id].index[0]
    similarity_scores = similarity_matrix[game_idx]
    similar_indices = similarity_scores.argsort()[::-1][1:n_recommendations+1]
    
    recommendations = []
    for idx in similar_indices:
        game_id = games_df.iloc[idx]['app_id']
        title = games_df.iloc[idx]['title']
        similarity = similarity_scores[idx]
        recommendations.append({
            'game_id': game_id,
            'title': title,
            'similarity_score': similarity
        })
    
    return recommendations

Simple Collaborative System

In [5]:
def get_collaborative_recommendations(user_id, recommendations_df, games_df, n_recommendations=5):
    user_games = recommendations_df[recommendations_df['user_id'] == user_id]
    user_positive_games = user_games[user_games['is_recommended'] == 1]['app_id'].tolist()
    
    if not user_positive_games:
        return []
    
    similar_users = recommendations_df[
        (recommendations_df['app_id'].isin(user_positive_games)) & 
        (recommendations_df['is_recommended'] == 1) &
        (recommendations_df['user_id'] != user_id)
    ]['user_id'].value_counts().head(5).index
    
    game_scores = {}
    for similar_user in similar_users:
        similar_user_games = recommendations_df[
            (recommendations_df['user_id'] == similar_user) & 
            (recommendations_df['is_recommended'] == 1)
        ]['app_id'].tolist()
        
        for game_id in similar_user_games:
            if game_id not in user_positive_games:
                if game_id not in game_scores:
                    game_scores[game_id] = 0
                game_scores[game_id] += 1
    
    recommendations = []
    for game_id, score in sorted(game_scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]:
        game_title = games_df[games_df['app_id'] == game_id]['title'].values[0]
        recommendations.append({
            'game_id': game_id,
            'title': game_title,
            'collab_score': score
        })
    
    return recommendations

Combining Content-Based and Collaborative Systems to build a Hybrid System

In [6]:
def get_hybrid_recommendations(user_id, game_id, similarity_matrix, games_df, recommendations_df, 
                             n_recommendations=5, content_weight=0.6, collab_weight=0.4):

    # Get content-based recommendations
    content_recs = get_content_recommendations(game_id, similarity_matrix, games_df, n_recommendations*2)
    content_scores = {rec['game_id']: rec['similarity_score'] for rec in content_recs}
    
    # Get collaborative recommendations
    collab_recs = get_collaborative_recommendations(user_id, recommendations_df, games_df, n_recommendations*2)
    collab_scores = {rec['game_id']: rec['collab_score'] for rec in collab_recs}
    
    # Combine scores
    all_game_ids = set(content_scores.keys()) | set(collab_scores.keys())
    hybrid_scores = {}
    
    # Min-max scaling for each set of scores
    if content_scores:
        max_content = max(content_scores.values())
        min_content = min(content_scores.values())
        content_range = max_content - min_content
    
    if collab_scores:
        max_collab = max(collab_scores.values())
        min_collab = min(collab_scores.values())
        collab_range = max_collab - min_collab
    
    for game_id in all_game_ids:
        # Normalize content score
        if game_id in content_scores:
            norm_content = (content_scores[game_id] - min_content) / content_range if content_range > 0 else 0
        else:
            norm_content = 0
            
        # Normalize collab score
        if game_id in collab_scores:
            norm_collab = (collab_scores[game_id] - min_collab) / collab_range if collab_range > 0 else 0
        else:
            norm_collab = 0
            
        # Calculate hybrid score
        hybrid_scores[game_id] = (content_weight * norm_content + collab_weight * norm_collab)
    
    # Sort by hybrid score and get top N
    sorted_games = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    
    # Format recommendations
    recommendations = []
    for game_id, score in sorted_games:
        title = games_df[games_df['app_id'] == game_id]['title'].iloc[0]
        recommendations.append({
            'game_id': game_id,
            'title': title
        })
    
    return recommendations

In [7]:
def create_cold_start_split(recommendations_df, test_size=0.15, val_size=0.15,
                            timestamp_col='date', cold_start_user_frac=0.01, seed=42):
    import numpy as np
    np.random.seed(seed)

    # Sort by timestamp if available
    if timestamp_col in recommendations_df.columns:
        recommendations_df = recommendations_df.sort_values(timestamp_col)

    # Identify cold-start users and items
    all_users = recommendations_df['user_id'].unique()
    all_items = recommendations_df['app_id'].unique()

    n_cold_users = int(len(all_users) * cold_start_user_frac)

    cold_users = np.random.choice(all_users, size=n_cold_users, replace=False)

    # Cold-start test data
    cold_user_df = recommendations_df[recommendations_df['user_id'].isin(cold_users)]

    # Remove cold users/items from the remaining dataset
    warm_df = recommendations_df[
        ~recommendations_df['user_id'].isin(cold_users)
    ]

    # Now split warm_df by user chronologically
    user_groups = warm_df.groupby('user_id')

    train_data = []
    val_data = []
    test_data = []

    for user_id, user_data in user_groups:
        n = len(user_data)
        if n < 3:
            continue

        n_test = max(1, int(n * test_size))
        n_val = max(1, int(n * val_size))

        user_train = user_data.iloc[:-n_test-n_val]
        user_val = user_data.iloc[-n_test-n_val:-n_test]
        user_test = user_data.iloc[-n_test:]

        train_data.append(user_train)
        val_data.append(user_val)
        test_data.append(user_test)

    train_df = pd.concat(train_data)
    val_df = pd.concat(val_data)
    test_df = pd.concat(test_data)

    return train_df, val_df, test_df, cold_user_df


train_df, val_df, test_df, cold_user_df = create_cold_start_split(recommendations_df)

print(f"Train Set: {len(train_df)} Recommendations")
print(f"Validation Set: {len(val_df)} Recommendations")
print(f"Test Set: {len(test_df)} Recommendations")
print(f"Cold-Start Test Set: {len(cold_user_df)} Users")


Train Set: 77464 Recommendations
Validation Set: 35794 Recommendations
Test Set: 35794 Recommendations
Cold-Start Test Set: 10446 Users


In [8]:
# Get a random user who has recommended games (is_recommended == 1)
users_with_recommendations = test_df[test_df['is_recommended'] == 1]['user_id'].unique()
sample_user = random.choice(users_with_recommendations)

# Get a random game that this user has recommended
user_recommended_games = test_df[(test_df['user_id'] == sample_user) & 
                               (test_df['is_recommended'] == 1)]['app_id'].values
sample_game = random.choice(user_recommended_games)
sample_game_title = games_df[games_df['app_id'] == sample_game]['title'].iloc[0]

print(f"Random User ID: {sample_user}")
print(f"Random Game ID: {sample_game}")
print(f"Random Game Title: {sample_game_title}\n")

print("Content-Based Recommendations:")
content_recs = get_content_recommendations(sample_game, similarity_matrix, games_df)
for i, rec in enumerate(content_recs, 1):
    print(f"{i}. {rec['title']} (Similarity: {rec['similarity_score']:.4f})")

print("\nCollaborative Recommendations:")
collab_recs = get_collaborative_recommendations(sample_user, recommendations_df, games_df)
for i, rec in enumerate(collab_recs, 1):
    print(f"{i}. {rec['title']} (Score: {rec['collab_score']})")

print("\nHybrid Recommendations:")
hybrid_recs = get_hybrid_recommendations(sample_user, sample_game, similarity_matrix, games_df, recommendations_df)
for i, rec in enumerate(hybrid_recs, 1):
    print(f"{i}. {rec['title']}")

Random User ID: 9699767
Random Game ID: 1333910
Random Game Title: Sizeable

Content-Based Recommendations:
1. Path of Giants (Similarity: 0.5250)
2. Mystery Hotel - Hidden Object Detective Game (Similarity: 0.5132)
3. Nubarron: The adventure of an unlucky gnome (Similarity: 0.5022)
4. Bunny's Maze (Similarity: 0.4798)
5. Magic Lessons in Wand Valley - a jigsaw puzzle tale (Similarity: 0.4734)

Collaborative Recommendations:
1. The Pedestrian (Score: 5)
2. There Is No Game: Wrong Dimension (Score: 4)
3. Carto (Score: 4)
4. WHAT THE GOLF? (Score: 4)
5. Spiritfarer®: Farewell Edition (Score: 3)

Hybrid Recommendations:
1. Path of Giants
2. Mystery Hotel - Hidden Object Detective Game
3. Nubarron: The adventure of an unlucky gnome
4. The Pedestrian
5. Bunny's Maze


In [9]:
def calculate_metrics(recommended_items, relevant_items, k):

    recommended_items = recommended_items[:k]
    relevant_items = set(relevant_items)  # Convert to set 
    
    # Precision@k
    hits = len(set(recommended_items) & relevant_items)
    precision = hits / k if k > 0 else 0
    
    # Recall@k
    recall = hits / len(relevant_items) if len(relevant_items) > 0 else 0
    
    # F1@k
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # NDCG@k calculation
    dcg = 0
    idcg = 0
    
    for i, item in enumerate(recommended_items):
        rel = 1 if item in relevant_items else 0
        dcg += rel / np.log2(i + 2)  
    
    n_rel = min(len(relevant_items), k)
    for i in range(n_rel):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ndcg': ndcg
    }

In [10]:
def evaluate_content_based_system(test_df, similarity_matrix, games_df, k=5):
    """
    Evaluate content-based system using unique games from test set, handling cold start cases
    """
    metrics_sum = {'precision': 0, 'recall': 0, 'f1': 0, 'ndcg': 0}
    total_users = 0
    
    # Get user-game pairs where the game was recommended (is_recommended == 1)
    positive_interactions = test_df[test_df['is_recommended'] == 1]
    
    # Group by user to get their liked games
    user_liked_games = positive_interactions.groupby('user_id')['app_id'].agg(list).to_dict()
    
    # Get all unique users
    all_users = test_df['user_id'].unique()
    
    print(f"Total users to evaluate: {len(all_users)}")
    
    for user_id in tqdm(all_users, desc='Content-Based Evaluation'):
        if user_id in user_liked_games:
            # Warm start case
            user_games = user_liked_games[user_id]
            if len(user_games) < 2:
                continue
                
            # Use one random game as input
            input_game = random.choice(user_games)
            # Use remaining games as ground truth
            test_games = set(user_games) - {input_game}
            
            try:
                recs = get_content_recommendations(input_game, similarity_matrix, games_df, n_recommendations=k)
                rec_ids = [rec['game_id'] for rec in recs]
                
                metrics = calculate_metrics(rec_ids, test_games, k)
                for metric, value in metrics.items():
                    metrics_sum[metric] += value
                total_users += 1
                
            except Exception as e:
                print(f"Error for user {user_id}: {str(e)}")
                continue
        else:
            # Cold start case - use genre/tag based recommendations
            # Get most diverse games based on tags and high ratings
            games_with_tags = games_df[games_df['tags'].notna()]
            diverse_games = (games_with_tags
                           .sort_values(['rating', 'user_reviews'], ascending=[False, False])
                           .drop_duplicates(subset=['tags'], keep='first')
                           .head(k))
            
            rec_ids = diverse_games['app_id'].tolist()
            test_games = set()  # No ground truth for cold start users
            
            metrics = calculate_metrics(rec_ids, test_games, k)
            for metric, value in metrics.items():
                metrics_sum[metric] += value
            total_users += 1
    
    # Calculate averages
    avg_metrics = {
        metric: value/total_users if total_users > 0 else 0 
        for metric, value in metrics_sum.items()
    }
    
    print("\nContent-Based Metrics:")
    for metric, value in avg_metrics.items():
        print(f"{metric.upper()}@{k}: {value:.4f}")
    
    return avg_metrics

In [11]:
def evaluate_collaborative_system(train_df, test_df, games_df, k=5):
    """
    Evaluate collaborative system, handling cold start cases
    """
    metrics_sum = {'precision': 0, 'recall': 0, 'f1': 0, 'ndcg': 0}
    total_users = 0
    
    # Get test set ground truth
    print("Building test set ground truth...")
    test_user_likes = defaultdict(set)
    for _, row in tqdm(test_df[test_df['is_recommended'] == 1].iterrows(), 
                      desc='Processing test data'):
        test_user_likes[row['user_id']].add(row['app_id'])

    # Pre-calculate popularity scores for cold start cases
    popularity_df = train_df[train_df['is_recommended'] == 1].copy()
    game_popularity = popularity_df.groupby('app_id').agg({
        'user_id': 'count',  # number of recommendations
    }).reset_index()
    
    # Merge with game information
    game_popularity = game_popularity.merge(
        games_df[['app_id', 'rating', 'user_reviews']], 
        on='app_id', 
        how='left'
    )
    
    # Calculate weighted popularity score
    game_popularity['popularity_score'] = (
        0.4 * game_popularity['user_id'] / game_popularity['user_id'].max() +
        0.3 * game_popularity['rating'] / game_popularity['rating'].max() +
        0.3 * game_popularity['user_reviews'] / game_popularity['user_reviews'].max()
    )
    
    # Sort by popularity score
    popular_games = game_popularity.sort_values('popularity_score', ascending=False)['app_id'].tolist()

    # Get all unique users from test set
    all_users = test_df['user_id'].unique()
    pbar = tqdm(total=len(all_users), desc='Collaborative Evaluation')

    for user_id in all_users:
        # Try to get collaborative recommendations
        recs = get_collaborative_recommendations(user_id, train_df, games_df, n_recommendations=k)
        
        if not recs:  # Cold start case
            # Use weighted popularity recommendations
            recommended_games = popular_games[:k]
        else:
            recommended_games = [rec['game_id'] for rec in recs]

        relevant_games = test_user_likes[user_id]
        metrics = calculate_metrics(recommended_games, relevant_games, k)
        
        for metric, value in metrics.items():
            metrics_sum[metric] += value
        total_users += 1
        
        pbar.update(1)

    pbar.close()

    avg_metrics = {
        metric: value/total_users if total_users > 0 else 0 
        for metric, value in metrics_sum.items()
    }
    
    print("\nCollaborative Filtering Metrics:")
    for metric, value in avg_metrics.items():
        print(f"{metric.upper()}@{k}: {value:.4f}")
    
    return avg_metrics

In [12]:
def evaluate_hybrid_system(test_df, similarity_matrix, games_df, recommendations_df, k=5, 
                         content_weight=0.5, collab_weight=0.5):
    """
    Evaluate hybrid system, handling cold start cases with adaptive weights
    """
    metrics_sum = {'precision': 0, 'recall': 0, 'f1': 0, 'ndcg': 0}
    total_users = 0
    
    # Get test set ground truth
    test_user_likes = defaultdict(set)
    for _, row in test_df[test_df['is_recommended'] == 1].iterrows():
        test_user_likes[row['user_id']].add(row['app_id'])
    
    # Get all unique users
    all_users = test_df['user_id'].unique()
    
    for user_id in tqdm(all_users, desc=f'Hybrid Evaluation (w_content={content_weight:.1f})'):
        # Get user's interaction history
        user_train_games = recommendations_df[
            (recommendations_df['user_id'] == user_id) & 
            (recommendations_df['is_recommended'] == 1)
        ]['app_id'].tolist()
        
        if not user_train_games:  # Cold start user
            # Adjust weights to rely more on content-based
            local_content_weight = 1
            local_collab_weight = 0
            
            # Get a diverse popular game as input
            popular_games = games_df.sort_values(['rating', 'user_reviews'], ascending=[False, False])
            diverse_games = popular_games.drop_duplicates(subset=['tags'], keep='first')
            input_game = diverse_games['app_id'].iloc[0]
        else:
            # Regular weights for warm start users
            local_content_weight = content_weight
            local_collab_weight = collab_weight
            input_game = random.choice(user_train_games)
            
        relevant_games = test_user_likes[user_id]
        
        try:
            recommendations = get_hybrid_recommendations(
                user_id=user_id,
                game_id=input_game,
                similarity_matrix=similarity_matrix,
                games_df=games_df,
                recommendations_df=recommendations_df,
                n_recommendations=k,
                content_weight=local_content_weight,
                collab_weight=local_collab_weight
            )
            
            rec_ids = [rec['game_id'] for rec in recommendations]
            metrics = calculate_metrics(rec_ids, relevant_games, k)
            
            for metric, value in metrics.items():
                metrics_sum[metric] += value
            total_users += 1
            
        except Exception as e:
            print(f"Error for user {user_id}: {str(e)}")
            continue
    
    # Calculate averages
    avg_metrics = {
        metric: value/total_users if total_users > 0 else 0 
        for metric, value in metrics_sum.items()
    }
    
    return avg_metrics

In [13]:
# Evaluate on different splits
print("Evaluating Regular Users...")
hybrid_metrics = evaluate_hybrid_system(test_df, similarity_matrix, games_df, recommendations_df, k=5)

print("\nEvaluating Cold Start Users...")
cold_user_hybrid_metrics = evaluate_hybrid_system(cold_user_df, similarity_matrix, games_df, recommendations_df, k=5)

Evaluating Regular Users...


Hybrid Evaluation (w_content=0.5): 100%|██████████| 34602/34602 [16:11<00:00, 35.60it/s]



Evaluating Cold Start Users...


Hybrid Evaluation (w_content=0.5): 100%|██████████| 8390/8390 [02:33<00:00, 54.65it/s]


In [16]:
scenarios = {
    'Regular Users': hybrid_metrics,
    'Cold Start Users': cold_user_hybrid_metrics
}

for scenario, metrics in scenarios.items():
    print(f"\n{scenario}:")
    print("-" * 30)
    for metric in ['precision', 'recall', 'f1', 'ndcg']:
        print(f"\n{metric.upper()}@5:")
        print(f"Hybrid: {metrics[metric]:.4f}")


Regular Users:
------------------------------

PRECISION@5:
Hybrid: 0.0020

RECALL@5:
Hybrid: 0.0096

F1@5:
Hybrid: 0.0032

NDCG@5:
Hybrid: 0.0064

Cold Start Users:
------------------------------

PRECISION@5:
Hybrid: 0.0009

RECALL@5:
Hybrid: 0.0018

F1@5:
Hybrid: 0.0011

NDCG@5:
Hybrid: 0.0016
